# 手动流水同步

## 概述

2.4节中CopyIn→Compute→CopyOut三步是串行执行的——搬运时计算单元闲置，硬件利用率低。本节介绍如何通过**双缓冲（Double Buffer）**和**手动流水同步**机制，让搬入、计算、搬出三步重叠执行，大幅提升算子性能。

### 学习目标

1. 理解AI Core的指令队列模型和并行机制；
2. 掌握双缓冲（Double Buffer）的工作原理；
3. 掌握`asc.set_flag`/`asc.wait_flag`同步接口和`HardEvent`枚举；
4. 能阅读和分析双缓冲流水同步代码。


In [ ]:
# 环境初始化
!mkdir -p Sources/02.05

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("环境初始化完成")

---
# 1. 双缓冲机制

## 1.1 AI Core指令队列模型

昇腾AI Core内部有多条独立的指令队列，矢量算子开发中主要涉及三类：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">指令队列</th>
      <th align="left">缩写</th>
      <th align="left">职责</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>MTE2</td><td>Memory Transfer Engine 2</td><td>GM → LM 数据搬入（CopyIn）</td></tr>
    <tr><td>Vector (V)</td><td>Vector</td><td>矢量计算（Compute）</td></tr>
    <tr><td>MTE3</td><td>Memory Transfer Engine 3</td><td>LM → GM 数据搬出（CopyOut）</td></tr>
  </tbody>
</table>

这三条指令队列可以**并行执行**——当MTE2在搬运下一块数据时，Vector可以同时计算当前块数据，MTE3可以搬出上一块的结果。这就是流水并行的硬件基础。

## 1.2 双缓冲原理

双缓冲使用两块缓冲区（BUFFER_NUM=2），当一块缓冲区正在计算时，另一块缓冲区可以同时进行数据搬运：

<img src="./images/double_buffer_pipeline.png" alt="双缓冲流水并行时序" width="800px">

图中Buf0和Buf1交替使用，CopyIn、Compute、CopyOut三级流水在不同缓冲区上重叠执行。

### 1.3 时序图解读

上图中，横轴为时间，纵轴为三个指令队列（CopyIn/Compute/CopyOut）：

- **t=0**：Buf0开始CopyIn（搬入第0块数据）
- **t=1**：Buf0开始Compute（计算第0块），同时Buf1开始CopyIn（搬入第1块）——**流水重叠开始**
- **t=2**：Buf0开始CopyOut（搬出第0块结果），同时Buf1开始Compute（计算第1块），Buf0开始CopyIn第2块
- **后续**：三步流水持续重叠执行，直到所有数据处理完成

关键点：如果没有双缓冲，CopyIn→Compute→CopyOut必须串行执行，每个数据块耗时3T；使用双缓冲后，从第2个数据块开始，三步重叠执行，平均每块耗时约1T。


---
# 2. 同步接口

虽然不同指令队列可以并行，但它们之间存在数据依赖：Compute必须等CopyIn完成才能读数据，CopyOut必须等Compute完成才能搬结果。pyasc通过`asc.set_flag`/`asc.wait_flag`建立同步关系。

## 2.1 HardEvent事件枚举

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">事件类型</th>
      <th align="left">同步方向</th>
      <th align="left">含义</th>
    </tr>
  </thead>
  <tbody>
    <tr><td><code>MTE2_V</code></td><td>搬入 → 计算</td><td>确保数据搬入完成后才开始计算</td></tr>
    <tr><td><code>V_MTE3</code></td><td>计算 → 搬出</td><td>确保计算完成后才开始搬出</td></tr>
    <tr><td><code>MTE3_MTE2</code></td><td>搬出 → 搬入</td><td>确保数据搬出完成后才复用缓冲区</td></tr>
  </tbody>
</table>

## 2.2 同步接口使用

```python
# 设置事件标志（通知下游可以执行）
asc.set_flag(asc.HardEvent.MTE2_V, buf_id)
# 等待事件标志（阻塞直到上游完成）
asc.wait_flag(asc.HardEvent.MTE2_V, buf_id)
```

其中`buf_id = i % BUFFER_NUM`用于在双缓冲之间切换，确保同步事件与正确的缓冲区对应。


### 2.3 为什么循环次数是 TILE_NUM * BUFFER_NUM？

单缓冲模式下，每个缓冲区处理 `TILE_NUM` 个数据块，循环 `TILE_NUM` 次。

双缓冲模式下，两个缓冲区交替使用，总共需要处理 `TILE_NUM` 个数据块。但由于双缓冲的流水特性，每个缓冲区实际处理 `TILE_NUM / 2` 块，两个缓冲区各处理一半，循环总次数为 `TILE_NUM * BUFFER_NUM / 2 * 2 = TILE_NUM * BUFFER_NUM`。

简化理解：双缓冲使循环次数翻倍，是因为两个缓冲区交替处理，每个数据块都需要经历CopyIn→Compute→CopyOut三步。

### 2.4 event_id机制

`buf_id`（0或1）作为`set_flag`/`wait_flag`的第二个参数（event_id），确保两个缓冲区的同步事件互不干扰：
- Buf0的同步事件使用event_id=0
- Buf1的同步事件使用event_id=1

这样Buf0的`wait_flag(MTE2_V, 0)`不会误等Buf1的`set_flag(MTE2_V, 1)`。


---
# 3. 流水同步分析

## 3.1 同步点分析表

Add样例在每个循环迭代中插入三对同步事件，对应三级流水的三处数据依赖：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">同步点</th>
      <th align="left">指令顺序</th>
      <th align="left">保护的数据</th>
      <th align="left">如果缺失，可能发生什么</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>MTE2_V</code></td>
      <td><code>data_copy</code> → <code>asc.add</code></td>
      <td>VECIN中的输入数据</td>
      <td>Vector可能读取到尚未搬入完成的数据</td>
    </tr>
    <tr>
      <td><code>V_MTE3</code></td>
      <td><code>asc.add</code> → <code>data_copy</code></td>
      <td>VECOUT中的计算结果</td>
      <td>MTE3可能搬出未完成计算的数据</td>
    </tr>
    <tr>
      <td><code>MTE3_MTE2</code></td>
      <td><code>data_copy</code>(Out) → <code>data_copy</code>(In)</td>
      <td>缓冲区复用安全</td>
      <td>下一次CopyIn可能覆盖尚未搬出的结果</td>
    </tr>
  </tbody>
</table>

## 3.2 流水循环代码结构

```python
for i in range(TILE_NUM * BUFFER_NUM):
    buf_id = i % BUFFER_NUM

    # Step 1: CopyIn
    asc.data_copy(x_local[buf_id * tile_length:], x_gm[i * tile_length:], tile_length)
    asc.data_copy(y_local[buf_id * tile_length:], y_gm[i * tile_length:], tile_length)
    asc.set_flag(asc.HardEvent.MTE2_V, buf_id)
    asc.wait_flag(asc.HardEvent.MTE2_V, buf_id)

    # Step 2: Compute
    asc.add(z_local[buf_id * tile_length:], x_local[buf_id * tile_length:],
            y_local[buf_id * tile_length:], tile_length)
    asc.set_flag(asc.HardEvent.V_MTE3, buf_id)
    asc.wait_flag(asc.HardEvent.V_MTE3, buf_id)

    # Step 3: CopyOut
    asc.data_copy(z_gm[i * tile_length:], z_local[buf_id * tile_length:], tile_length)
    asc.set_flag(asc.HardEvent.MTE3_MTE2, buf_id)
    asc.wait_flag(asc.HardEvent.MTE3_MTE2, buf_id)
```

> **注意：** 循环次数为`TILE_NUM * BUFFER_NUM`，因为双缓冲需要循环次数翻倍。


---
# 4. 运行Add样例分析流水

让我们运行Add样例，观察手动流水同步的执行。阅读代码时请重点关注：`set_flag`/`wait_flag`的成对使用，以及`buf_id`作为event_id的作用。


In [ ]:
# 运行Add样例
!python3 ./src/add.py -r NPU

In [ ]:
# 查看Add样例完整代码，重点关注同步接口
!cat ./src/add.py

---
# 4. 串行 vs 双缓冲性能对比

下图对比了串行执行和双缓冲流水执行的时间差异：

<img src="./images/serial_vs_double_buffer.png" alt="串行vs双缓冲对比" width="800px">

在串行模式下，每个数据块的CopyIn、Compute、CopyOut依次执行，耗时3T；N个数据块总耗时3N×T。

在双缓冲模式下，从第2个数据块开始三步重叠执行，N个数据块总耗时约(N+2)×T，流水效率提升接近3倍。


---
# 5. 小结

- **双缓冲（Double Buffer）**：基于MTE2/Vector/MTE3三条指令队列的独立性，将数据一分为二，实现CopyIn→Compute→CopyOut流水并行
- **同步接口**：`asc.set_flag`/`asc.wait_flag`建立指令队列间的数据依赖关系
- **HardEvent枚举**：`MTE2_V`（搬入→计算）、`V_MTE3`（计算→搬出）、`MTE3_MTE2`（搬出→复用缓冲区）
- **buf_id**：`i % BUFFER_NUM`在双缓冲之间切换，作为event_id确保同步事件与正确缓冲区对应

> **提示：** 手动同步需要开发者管理所有同步事件，代码较为复杂。第3章将介绍使用TPipe/TQue框架自动管理同步，大大简化代码。


---

## 课后实践

**实践任务：** 分析Add样例的流水同步代码，回答以下问题：

1. 如果将`BUFFER_NUM`从2改为1（单缓冲），流水循环次数应该是多少？还能实现流水并行吗？
2. `buf_id = i % BUFFER_NUM`的作用是什么？当`i=0`和`i=1`时，`buf_id`分别是多少？
3. 为什么需要`MTE3_MTE2`同步事件？如果去掉这个同步会导致什么问题？

<details>
<summary>点击查看参考答案</summary>

1. 单缓冲时循环次数为`TILE_NUM`（不需要翻倍）。单缓冲无法实现流水并行，因为同一块缓冲区不能同时进行搬入和计算。
2. `buf_id`用于在双缓冲之间切换。`i=0`时`buf_id=0`，`i=1`时`buf_id=1`，`i=2`时`buf_id=0`，以此类推。
3. `MTE3_MTE2`同步确保数据搬出完成后才复用该缓冲区进行下一次搬入。如果去掉，可能导致搬入覆盖了还未搬出的数据，造成数据错误。

</details>